# Fill nodata with 0

- Based on a spatial vector context fills the nodata values as 0


In [ ]:
import os
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from shapely.geometry import box
from sqlalchemy import create_engine, text

In [7]:
def load_vector_layer(db_name, user, password, host, port, table_name, schema='public', geom_col='geom'):
    """
    Connects to a PostGIS-enabled PostgreSQL database and loads a vector layer as a GeoDataFrame.
    
    Parameters:
    - db_name (str): Name of the PostgreSQL database.
    - user (str): Database username.
    - password (str): Database password.
    - host (str): Host address (e.g., 'localhost' or IP).
    - port (int): Port number (e.g., 5432).
    - table_name (str): Name of the table (vector layer) to load.
    - schema (str): Optional. Database schema containing the table (default is 'public').

    Returns:
    - gpd.GeoDataFrame: A GeoDataFrame containing the vector layer.
    """
    try:
        # Use pg8000 (pure Python driver)
        conn_str = f"postgresql+pg8000://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        sql = text(f"SELECT * FROM {schema}.{table_name}")

        # Open a connection explicitly (SQLAlchemy 2.x requirement)
        with engine.connect() as conn:
            gdf = gpd.read_postgis(sql, conn, geom_col=geom_col)
        
        print(f"Successfully loaded {table_name} ({len(gdf)} features)")
        return gdf

    except Exception as e:
        print(f"Error loading vector layer: {e}")
        return None
    
def get_raster_file_list(path):
    """Get a list of the raster files inside the folder"""
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        if file.endswith(".tif") or file.endswith(".tiff"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list

In [ ]:
"""Load the spatial context input"""
gdf_reference = load_vector_layer(
    db_name='geoserver',
    user='geoserver',
    password='geoserver',
    host='192.168.250.100',
    port=5555,
    table_name='global_osm_coastline_6km_buffer_country_split_4326',
    schema='public'
)

Successfully loaded global_osm_coastline_6km_buffer_country_split_4326 (6928 features)


In [9]:
gdf_reference.head()

,id,geom,gid,tile_index_1d,tile_index_3d,territory1,iso_sov1
0,1,"MULTIPOLYGON (((-68.74533 -56.42005, -68.74500...",1,11992,1358,Chile,CHL
1,2,"MULTIPOLYGON (((-67.24486 -56.00141, -67.24706...",2,11993,1358,Chile,CHL
2,16,"MULTIPOLYGON (((-66.98173 -54.97317, -66.98100...",21,12714,1358,Chile,CHL
3,17,"MULTIPOLYGON (((-74.00467 -53.23788, -74.00470...",33,13066,1476,Chile,CHL
4,18,"MULTIPOLYGON (((-73.25807 -53.99607, -73.25803...",34,13067,1476,Chile,CHL


In [ ]:
rasters_path = r"Y:\z_resources\un_gbf\02_deliverable_maps\coastal_2020\final_maps"
output_base_path = r"Y:\z_resources\un_gbf\02_deliverable_maps\coastal_2020\final_maps\fixed_layers_2"

raster_files = get_raster_file_list(rasters_path)

In [ ]:
# --- LOAD VECTOR ---

# gdf_base = gdf_reference.copy()

for raster_path in raster_files:
    with rasterio.open(raster_path) as src:
        
        print (f"Working on {os.path.basename(raster_path)}")
        profile = src.profile
        nodata = src.nodata

        # ensure CRS match
        # gdf = gdf_base
        # if gdf.crs != src.crs:
        #     gdf = gdf.to_crs(src.crs)

        # prepare shapes once (no dissolve)
        shapes = [(geom, 1) for geom in gdf_reference.geom if geom is not None]

        # set output path
        output_path = os.path.join(output_base_path, os.path.basename(raster_path).replace(".tif", "_fixed.tif"))

        with rasterio.open(output_path, "w", **profile) as dst:

            # iterate over raster blocks/windows
            for idx, (ji, window) in enumerate(src.block_windows(1)):
                
                if idx == 0:
                    print("Reading first window...")
                
                # read data
                data = src.read(1, window=window)

                # build transform for this window
                transform = src.window_transform(window)
                
                if idx % 100 == 0:
                    print(f"Processing window #{idx} at block index {ji}...")

                # rasterize only this window
                mask = rasterize(
                    shapes,
                    out_shape=data.shape,
                    transform=transform,
                    fill=0,
                    dtype="uint8"
                )

                # detect nodata
                if nodata is not None:
                    nodata_mask = data == nodata
                else:
                    nodata_mask = np.isnan(data)

                # fill nodata inside geometry
                data[(mask == 1) & nodata_mask] = 0

                # write result
                dst.write(data, 1, window=window)
                
                if idx % 100 == 0:
                    print(f"Written window #{idx}")
                    
            print("Finished processing all windows.")

Working on global_gbfb1_coastal_protection_combined_delta_epsg4326_50m_2020_v2.tif
Reading first window...
Processing window #0 at block index (0, 0)...
Written window #0


In [11]:
raster_files[1:]

['Y:\\z_resources\\un_gbf\\02_deliverable_maps\\coastal_2020\\final_maps\\global_gbfb1_coastal_protection_demand_epsg4326_50m_2020_v2.tif',
 'Y:\\z_resources\\un_gbf\\02_deliverable_maps\\coastal_2020\\final_maps\\global_gbfb1_coastal_protection_supply_epsg4326_50m_2020_v2.tif',
 'Y:\\z_resources\\un_gbf\\02_deliverable_maps\\coastal_2020\\final_maps\\global_gbfb1_coastal_protection_total_new_epsg4326_50m_2020_v2.tif']

In [ ]:
# ----------------------------
# LOAD VECTOR ONCE (KEEP CLEAN)
# ----------------------------
gdf_base = gdf_reference.copy()

print("Vector loaded.")
print(f"Initial feature count: {len(gdf_base)}")


# ----------------------------
# LOOP OVER RASTERS
# ----------------------------
for r_idx, raster_path in enumerate(raster_files):

    print("\n" + "=" * 60)
    print(f"[{r_idx+1}/{len(raster_files)}] Processing raster:")
    print(os.path.basename(raster_path))
    print("=" * 60)

    with rasterio.open(raster_path) as src:

        profile = src.profile.copy()
        nodata = src.nodata

        # ensure CRS match (do NOT overwrite base gdf)
        gdf = gdf_base
        if gdf.crs != src.crs:
            print("Reprojecting vector to raster CRS...")
            gdf = gdf.to_crs(src.crs)

        print(f"Vector features ready: {len(gdf)}")

        # update profile for safe BigTIFF writing
        profile.update(
            tiled=True,
            compress="lzw"
        )

        output_path = os.path.join(
            output_base_path,
            os.path.basename(raster_path).replace(".tif", "_fixed.tif")
        )

        print(f"Output will be saved to: {output_path}")

        with rasterio.open(output_path, "w", **profile) as dst:

            print("Starting window processing...")

            # ----------------------------
            # WINDOW LOOP
            # ----------------------------
            for idx, (ji, window) in enumerate(src.block_windows(1)):

                if idx == 0:
                    print("Reading first window...")

                data = src.read(1, window=window)
                transform = src.window_transform(window)

                # window bbox for spatial filtering
                bounds = rasterio.windows.bounds(window, src.transform)
                bbox = box(*bounds)

                # filter geometries intersecting window
                local_geoms = [
                    (geom, 1)
                    for geom in gdf.geometry
                    if geom is not None and geom.intersects(bbox)
                ]

                if idx % 50 == 0:
                    print(f"\nWindow {idx}")
                    print(f"Block index: {ji}")
                    print(f"Active geometries in window: {len(local_geoms)}")

                # rasterize only if needed
                if local_geoms:
                    mask = rasterize(
                        local_geoms,
                        out_shape=data.shape,
                        transform=transform,
                        fill=0,
                        dtype="uint8"
                    )
                else:
                    mask = np.zeros(data.shape, dtype="uint8")

                # nodata detection
                if nodata is not None:
                    nodata_mask = data == nodata
                else:
                    nodata_mask = np.isnan(data)

                # apply fill rule
                filled_pixels = (mask == 1) & nodata_mask
                data[filled_pixels] = 0

                # write window
                dst.write(data, 1, window=window)

                if idx % 50 == 0:
                    print(f"Window {idx} written.")
                    print(f"Pixels filled in this window: {filled_pixels.sum()}")

            print("\nFinished raster:")
            print(os.path.basename(raster_path))

print("\nALL RASTERS COMPLETED.")

In [ ]:
def progress(msg):
    # prints on the same line every time.
    print(f"\r{msg}", end="", flush=True)


# Keep an untouched copy
gdf_base = gdf_reference.copy()

print(f"Loaded vector with {len(gdf_base)} features")

# Build spatial index (R-tree)
# This lets us quickly find geometries near a bounding box
print("Building spatial index...")
sindex = gdf_base.sindex
print("Spatial index ready.")


# Loop through all rasters
for r_idx, raster_path in enumerate(raster_files[1:]):

    print("\n" + "=" * 50)
    print(f"[{r_idx+1}/{len(raster_files)}] {os.path.basename(raster_path)}")

    with rasterio.open(raster_path) as src:

        # Copy metadata (size, dtype, CRS, etc.)
        profile = src.profile.copy()

        # Get nodata value (could be None or NaN)
        nodata = src.nodata

        # crs handling: We never modify the original GeoDataFrame
        gdf = gdf_base

        # Reproject vector if CRS doesn't match raster
        if gdf.crs != src.crs:
            print("Reprojecting vector to match raster CRS...")
            gdf = gdf.to_crs(src.crs)

            # Spatial index must be rebuilt after reprojection
            sindex = gdf.sindex

        # Improve output raster performance
        profile.update(
            tiled=True,   
            compress="deflate"
        )

        # Output path
        output_path = os.path.join(
            output_base_path,
            os.path.basename(raster_path).replace(".tif", "_fixed.tif")
        )

        # Count total windows (for progress display)
        total_windows = sum(1 for _ in src.block_windows(1))

        with rasterio.open(output_path, "w", **profile) as dst:

            # This avoids loading huge rasters into memory
            for idx, (ji, window) in enumerate(src.block_windows(1)):

                # Show progress in one updating line
                progress(f"Window {idx+1}/{total_windows}")

                # Read only this small chunk of the raster
                data = src.read(1, window=window)

                # force float so NaN is allowed
                data = data.astype("float32")

                # If this window has all no nodata, skip everything
                if not np.isnan(data).any():
                    dst.write(data, 1, window=window)
                    continue

                # Get transform specific to this window
                transform = src.window_transform(window)

                # Get bounding box of this raster window
                bounds = rasterio.windows.bounds(window, src.transform)

                # Use spatial index to get candidate geometries
                candidate_idx = list(sindex.intersection(bounds))

                # If nothing intersects → skip
                if not candidate_idx:
                    dst.write(data, 1, window=window)
                    continue

                # Convert bounds to actual geometry
                bbox_geom = box(*bounds)

                # Precise filtering (geometry intersection)
                local_geoms = [
                    (geom, 1)
                    for geom in gdf.geometry.iloc[candidate_idx]
                    if geom is not None and geom.intersects(bbox_geom)
                ]

                # If still empty → skip
                if not local_geoms:
                    dst.write(data, 1, window=window)
                    continue

                # Rasterize vector → mask
                mask = rasterize(
                    local_geoms,
                    out_shape=data.shape,
                    transform=transform,
                    fill=0,
                    dtype="uint8",
                    all_touched=True
                )

                # Detect nodata pixels
                nodata_mask = np.isnan(data)


                # Only fill pixels inside geometry (mask == 1) and are nodata
                data[(mask == 1) & nodata_mask] = 0.0

                # Write processed window
                dst.write(data, 1, window=window)

        print("\nFinished raster.")

print("\nAll done.")

Loaded vector with 6928 features
Building spatial index...
Spatial index ready.

[1/4] global_gbfb1_coastal_protection_demand_epsg4326_50m_2020_v2.tif
Window 3088674/3088674
Finished raster.

[2/4] global_gbfb1_coastal_protection_supply_epsg4326_50m_2020_v2.tif
Window 3088674/3088674
Finished raster.

[3/4] global_gbfb1_coastal_protection_total_new_epsg4326_50m_2020_v2.tif
Window 3088674/3088674
Finished raster.

All done.


In [12]:
print(np.isnan(data).sum())
print((data == nodata).sum() if nodata else "no nodata")

65536
0


In [13]:
src.dtypes

('float32',)

In [14]:

src.nodata

nan